In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch, torchvision
import torchvision.transforms as T
from torch import nn, optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader as DL
from torch.nn.utils import weight_norm as WN
import torch.nn.functional as F

import gc
import os
import cv2
from time import time
from glob import glob
import re
from pathlib import Path

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)



In [2]:
# Resolve data root robustly (kept) so we read the correct train/test and sample_submission.
_CANDIDATE_INPUT_DIRS = [
    "/kaggle/data/ranzcr-clip-catheter-line-classification",
    "/kaggle/input/ranzcr-clip-catheter-line-classification",
    "/kaggle/input/ranzcr-clip-catheter-line-classification/ranzcr-clip-catheter-line-classification",
    "/kaggle/data/ranzcr-clip-catheter-line-classification/ranzcr-clip-catheter-line-classification",
]

INPUT_DIR = None
for p in _CANDIDATE_INPUT_DIRS:
    if os.path.exists(p):
        INPUT_DIR = p
        break

if INPUT_DIR is None:
    raise FileNotFoundError(
        "Could not locate competition data directory. Tried:\n"
        + "\n".join(_CANDIDATE_INPUT_DIRS)
    )

TRAIN_IMG_DIR = f"{INPUT_DIR}/train/"
TEST_IMG_DIR = f"{INPUT_DIR}/test/"
SAMPLE_SUB_PATH = f"{INPUT_DIR}/sample_submission.csv"
TRAIN_CSV_PATH = f"{INPUT_DIR}/train.csv"

print(f"Using INPUT_DIR={INPUT_DIR}")




Using INPUT_DIR=/kaggle/data/ranzcr-clip-catheter-line-classification


In [3]:
def breaker():
    print("\n" + 50 * "-" + "\n")


def head(x, no_of_ele=5):
    print(x[:no_of_ele])


def getImages(file_path=None, file_names=None, size=None):
    images = []
    for name in file_names:
        image_path = os.path.join(file_path, f"{name}.jpg")
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Image not found or unreadable: {image_path}")
        if size:
            image = cv2.resize(
                image, dsize=(size, size), interpolation=cv2.INTER_LANCZOS4
            )
            images.append(image.reshape(size, size, 1))
        else:
            h, w = image.shape[:2]
            images.append(image.reshape(h, w, 1))
    return np.array(images)




In [4]:
def _extract_epoch_num(path: str):
    m = re.search(r"Epoch[_\- ]?(\d+)\.pt$", os.path.basename(path))
    return int(m.group(1)) if m else None


def _norm(p: str) -> str:
    return p.replace("\\", "/")


def _unwrap_state_dict(obj):
    if (
        isinstance(obj, dict)
        and "state_dict" in obj
        and isinstance(obj["state_dict"], dict)
    ):
        return obj["state_dict"]
    if isinstance(obj, dict) and "model" in obj and isinstance(obj["model"], dict):
        return obj["model"]
    return obj


def _normalize_keys(state: dict):
    keys = list(state.keys())
    if keys and all(k.startswith("model.") for k in keys):
        state = {k[len("model.") :]: v for k, v in state.items()}
        keys = list(state.keys())
    if keys and all(k.startswith("module.") for k in keys):
        state = {k[len("module.") :]: v for k, v in state.items()}
    return state


def _is_compatible_state_dict(model: nn.Module, state: dict) -> bool:
    try:
        msd = model.state_dict()
        for k, v in state.items():
            if k in msd and torch.is_tensor(v) and torch.is_tensor(msd[k]):
                if tuple(v.shape) != tuple(msd[k].shape):
                    return False
        overlap = sum(1 for k in state.keys() if k in msd)
        return overlap >= max(10, int(0.2 * len(msd)))
    except Exception:
        return False


# Change (score-relevant): prioritize checkpoints from rccl-384-train strongly and pick closest epoch to desired.
# This increases likelihood we actually load trained weights (vs falling back to priors) without changing model/inference.
def find_best_compatible_checkpoint(preferred_path: str, model: nn.Module, device):
    desired_epoch = _extract_epoch_num(preferred_path) if preferred_path else None
    base = os.path.basename(preferred_path) if preferred_path else None

    if preferred_path and os.path.exists(preferred_path):
        try:
            obj = torch.load(preferred_path, map_location=device)
            state = _normalize_keys(_unwrap_state_dict(obj))
            if isinstance(state, dict) and _is_compatible_state_dict(model, state):
                print(
                    f"Checkpoint search: using provided path (compatible): {preferred_path}"
                )
                return _norm(preferred_path)
        except Exception as e:
            print(
                f"Checkpoint search: provided path exists but could not be read: {repr(e)}"
            )

    search_roots = ["/kaggle/input", "/kaggle/data"]

    # First pass: only rccl-384-train (most likely correct)
    candidates = []
    for root in search_roots:
        for g in [
            f"{root}/**/rccl-384-train/**/Epoch_*.pt",
            f"{root}/**/rccl-384-train/**/epoch_*.pt",
        ]:
            for p in glob(g, recursive=True):
                if os.path.isfile(p) and _extract_epoch_num(p) is not None:
                    candidates.append(_norm(p))

    # Second pass: exact basename match (if provided)
    if base:
        for root in search_roots:
            for p in glob(f"{root}/**/{base}", recursive=True):
                if os.path.isfile(p):
                    candidates.append(_norm(p))

    # Third pass (fallback): any Epoch_*.pt (kept, but ranked worse due to longer path heuristic)
    for root in search_roots:
        for p in glob(f"{root}/**/Epoch_*.pt", recursive=True):
            if os.path.isfile(p) and _extract_epoch_num(p) is not None:
                candidates.append(_norm(p))

    seen = set()
    candidates = [p for p in candidates if not (p in seen or seen.add(p))]

    compatible = []
    for p in candidates:
        try:
            obj = torch.load(p, map_location=device)
            state = _unwrap_state_dict(obj)
            if not isinstance(state, dict):
                continue
            state = _normalize_keys(state)
            if _is_compatible_state_dict(model, state):
                e = _extract_epoch_num(p)
                # Prefer: closest epoch; then prefer paths containing rccl-384-train; then shorter path.
                has_rccl = 0 if ("/rccl-384-train/" in _norm(p)) else 1
                if desired_epoch is None or e is None:
                    rank = (10**9, has_rccl, len(p), p)
                else:
                    rank = (abs(e - desired_epoch), has_rccl, len(p), p)
                compatible.append((rank, p, e))
        except Exception:
            continue

    if compatible:
        compatible.sort(key=lambda x: x[0])
        print("Checkpoint search: found compatible checkpoints (top 5 by rank):")
        for r, p, e in compatible[:5]:
            print(f"  epoch={e}, rank={r[:3]}, path={p}")
        return compatible[0][1]

    return None




In [5]:
def load_state_dict_safely(model, ckpt_path, device):
    if ckpt_path is None:
        return False, "No checkpoint found."

    try:
        state = torch.load(ckpt_path, map_location=device)
        state = _unwrap_state_dict(state)

        if not isinstance(state, dict):
            return False, f"Checkpoint format not a state_dict: {ckpt_path}"

        state = _normalize_keys(state)

        if not _is_compatible_state_dict(model, state):
            return False, f"Incompatible checkpoint (shape/key mismatch): {ckpt_path}"

        model.load_state_dict(state, strict=True)
        return True, f"Loaded checkpoint: {ckpt_path}"
    except Exception as e:
        return False, f"Failed to load checkpoint ({ckpt_path}): {repr(e)}"




In [6]:
start_time = time()

ss = pd.read_csv(SAMPLE_SUB_PATH)
train_df = pd.read_csv(TRAIN_CSV_PATH)

# Change (score-relevant): competition requires 11 targets; always derive and submit all 11 in train.csv order.
ALL_TARGET_COLS = [
    c for c in train_df.columns if c not in ["StudyInstanceUID", "PatientID"]
]
assert (
    len(ALL_TARGET_COLS) == 11
), f"Expected 11 targets, got {len(ALL_TARGET_COLS)}: {ALL_TARGET_COLS}"

test_files = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
assert len(test_files) > 0, f"No test images found under {TEST_IMG_DIR}"
ts_img_names = np.array([Path(p).stem for p in test_files])

ts_images = getImages(TEST_IMG_DIR, ts_img_names, size=384)

breaker()
print("Time Taken to read data : {:.2f} minutes".format((time() - start_time) / 60))
print(f"Sample submission shape (may be truncated here): {ss.shape}")
print(f"Test images found: {len(ts_img_names)}")
print(
    f"Using target columns from train.csv ({len(ALL_TARGET_COLS)}): {ALL_TARGET_COLS}"
)
print(
    f"Submission target columns present in provided sample_submission: {list(ss.columns[1:])}"
)
breaker()





--------------------------------------------------

Time Taken to read data : 0.64 minutes
Sample submission shape (may be truncated here): (3009, 10)
Test images found: 3009
Using target columns from train.csv (11): ['ETT - Abnormal', 'ETT - Borderline', 'ETT - Normal', 'NGT - Abnormal', 'NGT - Borderline', 'NGT - Incompletely Imaged', 'NGT - Normal', 'CVC - Abnormal', 'CVC - Borderline', 'CVC - Normal', 'Swan Ganz Catheter Present']
Submission target columns present in provided sample_submission: ['ETT - Abnormal', 'ETT - Borderline', 'ETT - Normal', 'NGT - Abnormal', 'NGT - Borderline', 'NGT - Incompletely Imaged', 'NGT - Normal', 'CVC - Abnormal', 'CVC - Borderline']

--------------------------------------------------



In [7]:
class DS(Dataset):
    def __init__(this, X=None, y=None, transform=None, mode="train"):
        this.mode = mode
        this.transform = transform
        this.X = X
        if mode == "train":
            this.y = y

    def __len__(this):
        return this.X.shape[0]

    def __getitem__(this, idx):
        img = this.transform(this.X[idx])
        if this.mode == "train":
            return img, torch.FloatTensor(this.y[idx])
        else:
            return img




In [8]:
tfm = T.Compose([T.ToTensor()])

ts_data_setup = DS(X=ts_images, y=None, transform=tfm, mode="test")
ts_data = DL(
    ts_data_setup,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)




In [9]:
class CFG:
    tr_batch_size = 64
    ts_batch_size = 64

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    in_channels = 1

    OL = len(ALL_TARGET_COLS)

    def __init__(
        this, filter_sizes=[64, 128, 256, 512], HL=[2048], epochs=50, n_folds=5
    ):
        this.filter_sizes = filter_sizes
        this.HL = HL
        this.epochs = epochs
        this.n_folds = n_folds




In [10]:
class CNN(nn.Module):
    def __init__(
        this, in_channels=1, filter_sizes=None, HL=None, OL=None, use_DP=True, DP=0.50
    ):
        super(CNN, this).__init__()

        this.use_DP = use_DP

        this.DP_ = nn.Dropout(p=DP)
        this.MP_ = nn.MaxPool2d(kernel_size=2)

        this.CN1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=filter_sizes[0],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        this.BN1 = nn.BatchNorm2d(num_features=filter_sizes[0], eps=1e-5)

        this.CN2 = nn.Conv2d(
            in_channels=filter_sizes[0],
            out_channels=filter_sizes[1],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        this.BN2 = nn.BatchNorm2d(num_features=filter_sizes[1], eps=1e-5)

        this.CN3 = nn.Conv2d(
            in_channels=filter_sizes[1],
            out_channels=filter_sizes[2],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        this.BN3 = nn.BatchNorm2d(num_features=filter_sizes[2], eps=1e-5)

        this.CN4 = nn.Conv2d(
            in_channels=filter_sizes[2],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        this.BN4 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        this.CN5 = nn.Conv2d(
            in_channels=filter_sizes[3],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        this.BN5 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        this.CN6 = nn.Conv2d(
            in_channels=filter_sizes[3],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        this.BN6 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        this.CN7 = nn.Conv2d(
            in_channels=filter_sizes[3],
            out_channels=filter_sizes[3],
            kernel_size=3,
            stride=1,
            padding=1,
        )
        this.BN7 = nn.BatchNorm2d(num_features=filter_sizes[3], eps=1e-5)

        this.FC1 = nn.Linear(in_features=filter_sizes[3] * 3 * 3, out_features=HL[0])
        this.FC2 = nn.Linear(in_features=HL[0], out_features=OL)

    def getOptimizer(this, lr=1e-3, wd=0):
        return optim.Adam(this.parameters(), lr=lr, weight_decay=wd)

    def getStepLR(this, optimizer=None, step_size=5, gamma=0.1):
        return optim.lr_scheduler.StepLR(
            optimizer=optimizer, step_size=step_size, gamma=gamma
        )

    def getMultiStepLR(this, optimizer=None, milestones=None, gamma=0.1):
        return optim.lr_scheduler.MultiStepLR(
            optimizer=optimizer, milestones=milestones, gamma=gamma
        )

    def getPlateauLR(this, optimizer=None, patience=5, eps=1e-8):
        return optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=optimizer, patience=patience, eps=1e-8, verbose=True
        )

    def forward(this, x):
        if this.use_DP:
            x = F.relu(this.MP_(this.BN1(this.CN1(x))))
            x = F.relu(this.MP_(this.BN2(this.CN2(x))))
            x = F.relu(this.MP_(this.BN3(this.CN3(x))))
            x = F.relu(this.MP_(this.BN4(this.CN4(x))))
            x = F.relu(this.MP_(this.BN5(this.CN5(x))))
            x = F.relu(this.MP_(this.BN6(this.CN6(x))))
            x = F.relu(this.MP_(this.BN7(this.CN7(x))))

            x = x.view(x.shape[0], -1)

            x = F.relu(this.DP_(this.FC1(x)))
            x = this.FC2(x)
            return x
        else:
            x = F.relu(this.MP_(this.BN1(this.CN1(x))))
            x = F.relu(this.MP_(this.BN2(this.CN2(x))))
            x = F.relu(this.MP_(this.BN3(this.CN3(x))))
            x = F.relu(this.MP_(this.BN4(this.CN4(x))))
            x = F.relu(this.MP_(this.BN5(this.CN5(x))))
            x = F.relu(this.MP_(this.BN6(this.CN6(x))))
            x = F.relu(this.MP_(this.BN7(this.CN7(x))))

            x = x.view(x.shape[0], -1)

            x = F.relu(this.FC1(x))
            x = this.FC2(x)
            return x




In [11]:
def predict_(model=None, dataloader=None, device=None, path=None, out_dim=11):
    """
    Keep core inference semantics unchanged when a checkpoint is found (sigmoid over logits).
    If no compatible checkpoint is found/loaded, return None so caller can fall back safely.
    """
    ckpt = find_best_compatible_checkpoint(path, model, device) if path else None
    if ckpt is None:
        print(
            "WARNING: No compatible checkpoint found under /kaggle/input or /kaggle/data. "
            "Will fall back to per-label train priors."
        )
        return None

    ok, msg = load_state_dict_safely(model, ckpt, device)
    print(msg)
    if not ok:
        print(
            "WARNING: Checkpoint was found but failed to load. Falling back to priors."
        )
        return None

    model.to(device)
    model.eval()

    y_pred = torch.zeros(1, out_dim, device=device)

    for X in dataloader:
        X = X.to(device, non_blocking=True)
        with torch.no_grad():
            Pred = torch.sigmoid(model(X))
        y_pred = torch.cat((y_pred, Pred), dim=0)

    return y_pred[1:].detach().cpu().numpy()




In [12]:
cfg = CFG(filter_sizes=[64, 128, 256, 512], HL=[2048], epochs=30, n_folds=5)

model = CNN(
    in_channels=cfg.in_channels,
    filter_sizes=cfg.filter_sizes,
    HL=cfg.HL,
    OL=cfg.OL,
)

ckpt_path = "/kaggle/input/rccl-384-train/Epoch_14.pt"

y_pred = predict_(
    model=model, dataloader=ts_data, device=cfg.device, path=ckpt_path, out_dim=cfg.OL
)

# Model output column order (kept): must match the checkpoint training label order.
model_out_cols = [
    "ETT - Abnormal",
    "ETT - Borderline",
    "ETT - Normal",
    "NGT - Abnormal",
    "NGT - Borderline",
    "NGT - Incompletely Imaged",
    "NGT - Normal",
    "CVC - Abnormal",
    "CVC - Borderline",
    "CVC - Normal",
    "Swan Ganz Catheter Present",
]
col_to_idx = {c: i for i, c in enumerate(model_out_cols)}

# Change (score-relevant): always write the full 11-column competition schema from train.csv,
# not the truncated sample_submission in this environment, to avoid Kaggle treating missing labels as constants.
SUB_COLS = ["StudyInstanceUID"] + ALL_TARGET_COLS
SUB_TARGET_COLS = ALL_TARGET_COLS

# Row alignment: prefer official ordering from sample_submission if available, else from test folder.
if "StudyInstanceUID" not in ss.columns:
    raise ValueError("sample_submission.csv missing StudyInstanceUID column.")
sub_uids = ss["StudyInstanceUID"].astype(str).values
pred_uids = ts_img_names.astype(str)

uid_to_row = pd.Series(np.arange(len(pred_uids), dtype=np.int32), index=pred_uids)
row_idx = uid_to_row.reindex(sub_uids).to_numpy()
valid = ~pd.isna(row_idx)
row_idx = np.where(valid, row_idx.astype(np.int32), -1)

# Priors used only when checkpoint missing/unloaded or for any unmapped UIDs.
priors = train_df[ALL_TARGET_COLS].mean().values.astype(np.float32)
priors = np.clip(priors, 1e-6, 1 - 1e-6)
prior_map = {c: priors[i] for i, c in enumerate(ALL_TARGET_COLS)}

sub = pd.DataFrame({"StudyInstanceUID": sub_uids})

if y_pred is not None:
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)

for col in SUB_TARGET_COLS:
    base = np.full((len(sub_uids),), prior_map.get(col, 0.5), dtype=np.float32)
    if y_pred is not None and col in col_to_idx and col_to_idx[col] < y_pred.shape[1]:
        vi = np.where(row_idx >= 0)[0]
        base[vi] = y_pred[row_idx[vi], col_to_idx[col]].astype(np.float32)
    sub[col] = base

sub = sub[SUB_COLS]
sub.to_csv("./submission.csv", index=False)

# Validate output is a proper competition-format submission (12 cols = UID + 11 targets).
assert (
    list(sub.columns) == SUB_COLS
), "Submission columns must match required schema (UID + 11 targets)."
assert len(sub) == len(ss), "Row count must match sample_submission UID list."

print(sub.head(5))
print(f"Saved submission.csv with shape {sub.shape} and columns {list(sub.columns)}")
print("Prediction summary (per-column mean):")
print(sub[SUB_TARGET_COLS].mean().to_string())
if y_pred is None:
    print(
        "NOTE: submission.csv uses train priors because no compatible checkpoint was found/loaded."
    )

                                    StudyInstanceUID  ETT - Abnormal  \
0  1.2.826.0.1.3680043.8.498.25512976433640891933...        0.002807   
1  1.2.826.0.1.3680043.8.498.24449897997512078380...        0.002807   
2  1.2.826.0.1.3680043.8.498.38485493636649999035...        0.002807   
3  1.2.826.0.1.3680043.8.498.11901773728604504629...        0.002807   
4  1.2.826.0.1.3680043.8.498.11327663178389439022...        0.002807   

   ETT - Borderline  ETT - Normal  NGT - Abnormal  NGT - Borderline  \
0          0.037933      0.240194        0.009382          0.017988   
1          0.037933      0.240194        0.009382          0.017988   
2          0.037933      0.240194        0.009382          0.017988   
3          0.037933      0.240194        0.009382          0.017988   
4          0.037933      0.240194        0.009382          0.017988   

   NGT - Incompletely Imaged  NGT - Normal  CVC - Abnormal  CVC - Borderline  \
0                   0.090604      0.159193        0.106929  